# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:150%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Diamond Price Analysis & Prediction</p>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn import metrics
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline, make_pipeline
import shap

<a id="3"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Basic Exploration</p>


**Let's have a glimpse at the dataset.**

* **carat:** Weight(0.2 to 5.01) of the diamond
* **cut:** Quality of the cut (Fair, Good, Very Good, Premium, Ideal)
* **color:** Diamond colour from J(worst) to D(best)
* **clarity:** Measurement of how clear the diamond is (I1 (worst), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (best))
* **x:** Length(0 to 10.74) in mm 
* **y:** Width(0 to 58.9) in mm 
* **z:** Depth(0 to 31.8) in mm 
* **depth %:** The height of a diamond, measured from the culet to the table, divided by its average girdle diameter. Total depth percentage(43 to 79) = z/mean(x,y) = 2\*z/(x+y)
* **table:** Width(43 to 95) of the top of diamond relative to widest point 
* **price:** Price in US dollar

In [ ]:
diamonds = pd.read_csv('diamonds.csv',index_col=False)
diamonds.head().style.set_properties(**{"background-color": "#FDD667","color":"black","border": "1.5px solid black"})

In [ ]:
diamonds.info()

<a id="4"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Dataset Summary</p>

In [ ]:
diamonds.describe().style.set_properties(**{"background-color": "#FDD667","color":"black","border": "1.5px solid black"})

In [ ]:
diamonds.describe(include=object).T.style.set_properties(**{"background-color": "#FDD667","color":"black","border": "1.5px solid black"})

In [ ]:
diamonds['color'].value_counts()

In [ ]:
# Any nulls?
diamonds.isna().sum().to_frame().T.style.set_properties(**{"background-color": "#FDD667","color":"black","border": "1.5px solid black"})

In [ ]:
# The describe table finds values = 0 in x, y and z. How many?
len(diamonds.loc[(diamonds["x"] == 0) | (diamonds["y"] == 0) | (diamonds["z"] == 0)])

In [ ]:
diamonds.shape

In [ ]:
#dropping zeroes
diamonds = diamonds[(diamonds[["x", "y", "z"]] != 0).all(axis=1)].reset_index()
diamonds = diamonds.drop("index", axis=1)

In [ ]:
diamonds.shape # check that all 0's have been dropped


In [ ]:
diamonds.describe().style.set_properties(**{"background-color": "#FDD667","color":"black","border": "1.5px solid black"}) # this makes more sense now

**Insights:**

* Fixed the wrong values
* We will encode the categorical features into numerical ones later.

<a id="5"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Custom Palette For Visualization</p>

In [ ]:
sns.set_style("white")
sns.set(rc={"axes.facecolor":"#f2d4b1","figure.facecolor":"#f2d4b1","grid.color":"white"})
sns.set_context("poster",font_scale = .7)

palette = ["#c94727","#ea5b17","#e57716","#f2a324","#a2c0a6","#7ac0a8","#5e9786","#557260"]

sns.palplot(sns.color_palette(palette))
plt.show()

<a id="6"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Diamond Prices</p>

In [ ]:
print("Let's have a look at the distribution of prices :")
plt.subplots(figsize=(10, 4))
p = sns.histplot(diamonds["price"],color=palette[6],kde=True,bins=30,alpha=1,fill=True,edgecolor="black",linewidth=3)
p.axes.lines[0].set_color("orange")
p.axes.set_title("\nDiamonds Pricing Distribution\n",fontsize=25)
plt.ylabel("Count",fontsize=20)
plt.xlabel("\nPrice",fontsize=20)
plt.yscale("linear")
sns.despine(left=True, bottom=True)

plt.show()

- "Price", as expected, is skewed. There are few diamonds which are worth too much and a lot of diamonds with reasonably small prices.

<a id="7"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Diamond Cuts</p>

In [ ]:
print(f"Let's have a look at the ratio of diamond cuts:")
plt.subplots(figsize=(12, 12))

labels = "Ideal","Premium","Very Good","Good","Fair"
size = 0.5

wedges, texts, autotexts = plt.pie([diamonds["cut"].value_counts().values[0],
                                    diamonds["cut"].value_counts().values[1],
                                    diamonds["cut"].value_counts().values[2],
                                    diamonds["cut"].value_counts().values[3],
                                    diamonds["cut"].value_counts().values[4]],
                                    explode = (0,0,0,0,0),
                                    textprops=dict(size= 20, color= "white"),
                                    autopct="%.2f%%", 
                                    pctdistance = 0.72,
                                    radius=.9, 
                                    colors = ["#3f4f45","#5e9880","#f5a126","#ea5b17","#6c3938"], 
                                    shadow = False,
                                    wedgeprops=dict(width = size, edgecolor = "black", 
                                    linewidth = 4),
                                    startangle = 0)

plt.legend(wedges, labels, title="Diamonds Cut",loc="upper right", edgecolor = "black") #,bbox_to_anchor=(1, 0, 0.5, 1)
plt.title("\nDiamonds Cut Ratio",fontsize=25)
plt.show()

In [ ]:
print("On absolute values:")
plt.subplots(figsize=(20, 8))
p=sns.countplot(y=diamonds["cut"],order=diamonds["cut"].value_counts().index,palette=["#3f4f45","#5e9880","#f5a126","#ea5b17","#6c3938"], saturation=1, edgecolor = "#1c1c1c", linewidth = 5)
# p.axes.set_yscale("symlog")
p.axes.set_title("\nDiamond's Cut\n",fontsize=25)
p.axes.set_ylabel("Cut",fontsize=20)
p.axes.set_xlabel("\nTotal",fontsize=20)
p.axes.set_yticklabels(p.get_yticklabels(),rotation = 0)
for container in p.containers:
    p.bar_label(container,label_type="center",padding=6,size=25,color="black",rotation=0,
    bbox={"boxstyle": "round", "pad": 0.4, "facecolor": "#e0b583", "edgecolor": "#1c1c1c", "linewidth" : 4, "alpha": 1})


sns.despine(left=True, bottom=True)
plt.show()

**Insights:**

* Most of the diamonds have **Ideal Cut** with a ratio of **39.95%** followed by **Premium Cut** and **Very Good Cut**
* Only few have **Fair Cut** with a ratio of **2.98%**.

In [ ]:
print("Let's have a look at the price distribution of diamond cuts:")
plt.subplots(figsize=(25, 10))

p=sns.violinplot(x=diamonds["cut"],y=diamonds["price"],order=diamonds["cut"].value_counts().index,palette=["#3f4f45","#5e9880","#f5a126","#ea5b17","#6c3938"],saturation=1,linewidth=4,edgecolor="black")
p.axes.set_title("\nDiamond Cut pricing\n",fontsize=30)
p.axes.set_xlabel("\nCut",fontsize=25)
p.axes.set_ylabel("Price",fontsize=25)
p.set_yticklabels(p.get_yticks(), size = 20)
p.set_xticklabels(diamonds["cut"].value_counts().index, size = 25)

sns.despine(left=True, bottom=True)
plt.show()

**Insights:**

* Most of the diamonds with **Ideal Cut** costs in between **326** to **2500**
* Most of the diamonds with **Premium Cut** costs in between **326** to **5000**
* Most of the diamonds with **Very Good Cut** costs in between **336** to **4800**
* Most of the diamonds with **Good Cut** costs in between **327** to **4700**
* Most of the diamonds with **Fair Cut** costs in between **337** to **5000**

<a id="8"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Diamond Colors</p>

In [ ]:
diamonds["color"].value_counts().sort_index(ascending=True)

In [ ]:
print(f"Let's have a look at the ratio of diamond colors, which range from J (worst) to D (best):")
plt.subplots(figsize=(12, 12))

labels = "D (best)","E","F","G","H","I","J (worst)"
size = 0.5

wedges, texts, autotexts = plt.pie([diamonds["color"].value_counts().sort_index(ascending=True).values[0],
                                    diamonds["color"].value_counts().sort_index(ascending=True).values[1],
                                    diamonds["color"].value_counts().sort_index(ascending=True).values[2],
                                    diamonds["color"].value_counts().sort_index(ascending=True).values[3],
                                    diamonds["color"].value_counts().sort_index(ascending=True).values[4],
                                    diamonds["color"].value_counts().sort_index(ascending=True).values[5],
                                    diamonds["color"].value_counts().sort_index(ascending=True).values[6]],
                                    explode = (0,0,0,0,0,0,0),
                                    textprops=dict(size= 20, color= "white"),
                                    autopct="%.2f%%", 
                                    pctdistance = 0.72,
                                    radius=.9, 
                                    colors = ["#3f4f45","#5e9880","#a2c0a6","#f5a126","#b05f0d","#ea5b17","#6c3938"], 
                                    shadow = False,
                                    wedgeprops=dict(width = size, edgecolor = "black", 
                                    linewidth = 4),
                                    startangle = 70)

plt.legend(wedges, labels, title="Diamond Colors",loc="center left",bbox_to_anchor=(0.9, 0, 0.5, 1), edgecolor = "black")
plt.title("\nDiamond Color Ratio",fontsize=25)
plt.show()

In [ ]:
print("Let's have a look at the absolute values now:")
plt.subplots(figsize=(20, 8))
p=sns.countplot(y=diamonds["color"],order=diamonds["color"].value_counts().index,palette=["#3f4f45","#5e9880","#a2c0a6","#f5a126","#b05f0d","#ea5b17","#6c3938"], saturation=1, edgecolor = "#1c1c1c", linewidth = 5)
# p.axes.set_yscale("symlog")
p.axes.set_title("\nDiamond Colors\n",fontsize=25)
p.axes.set_ylabel("Color",fontsize=20)
p.axes.set_xlabel("\nTotal",fontsize=20)
p.axes.set_yticklabels(p.get_yticklabels(),rotation = 0)
for container in p.containers:
    p.bar_label(container,label_type="center",padding=6,size=25,color="black",rotation=0,
    bbox={"boxstyle": "round", "pad": 0.2, "facecolor": "#e0b583", "edgecolor": "#1c1c1c", "linewidth" : 4, "alpha": 1})


sns.despine(left=True, bottom=True)
plt.show()

In [ ]:
print("Let's have a look at the price distribution of diamond colors:")
plt.subplots(figsize=(25, 10))

p=sns.violinplot(x=diamonds["color"],y=diamonds["price"],order=diamonds["color"].value_counts().index,palette=["#3f4f45","#5e9880","#a2c0a6","#f5a126","#b05f0d","#ea5b17","#6c3938"],saturation=1,linewidth=4,edgecolor="black")
p.axes.set_title("\nDiamonds Color pricing\n",fontsize=30)
p.axes.set_xlabel("\nColor",fontsize=25)
p.axes.set_ylabel("Price",fontsize=25)
p.set_yticklabels(p.get_yticks(), size = 20)
p.set_xticklabels(diamonds["color"].value_counts().index, size = 25)

sns.despine(left=True, bottom=True)
plt.show()

<a id="9"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Diamond Clarities</p>

In [ ]:
print(f"Let's have a look at the ratio of diamond clarities. The range is (I1 (worst), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (best)):")
plt.subplots(figsize=(12, 12))

labels = "SI1","VS2","SI2","VS1","VVS2","VVS1","IF","I1"
size = 0.5

wedges, texts, autotexts = plt.pie([diamonds["clarity"].value_counts().values[0],
                                    diamonds["clarity"].value_counts().values[1],
                                    diamonds["clarity"].value_counts().values[2],
                                    diamonds["clarity"].value_counts().values[3],
                                    diamonds["clarity"].value_counts().values[4],
                                    diamonds["clarity"].value_counts().values[5],
                                    diamonds["clarity"].value_counts().values[6],
                                    diamonds["clarity"].value_counts().values[7]],
                                    explode = (0,0,0,0,0,0,0,0),
                                    textprops=dict(size= 20, color= "white"),
                                    autopct="%.2f%%", 
                                    pctdistance = 0.72,
                                    radius=.9, 
                                    colors = ["#3f4f45","#557260","#5e9880","#a2c0a6","#f5a126","#e57716","#ea5b17","#6c3938"], 
                                    shadow = False,
                                    wedgeprops=dict(width = size, edgecolor = "black", 
                                    linewidth = 4),
                                    startangle = 45)

plt.legend(wedges, labels, title="Diamond's Clarity",loc="center left",bbox_to_anchor=(0.9, 0, 0.5, 1), edgecolor = "black")
plt.title("\nDiamonds Clarity Ratio",fontsize=25)
plt.show()

In [ ]:
print("In absolute values, we get")
plt.subplots(figsize=(20, 8))
p=sns.countplot(y=diamonds["clarity"],order=diamonds["clarity"].value_counts().index,palette=["#3f4f45","#557260","#5e9880","#a2c0a6","#f5a126","#e57716","#ea5b17","#6c3938"], saturation=1, edgecolor = "#1c1c1c", linewidth = 5)
# p.axes.set_yscale("symlog")
p.axes.set_title("\nDiamond's Clarity\n",fontsize=25)
p.axes.set_ylabel("Clarity",fontsize=20)
p.axes.set_xlabel("\nTotal",fontsize=20)
p.axes.set_yticklabels(p.get_yticklabels(),rotation = 0)
for container in p.containers:
    p.bar_label(container,label_type="center",padding=6,size=22,color="black",rotation=0,
    bbox={"boxstyle": "round", "pad": 0.2, "facecolor": "#e0b583", "edgecolor": "#1c1c1c", "linewidth" : 3, "alpha": 1})


sns.despine(left=True, bottom=True)
plt.show()

**Insights:**

* Most of the diamonds have **SI1** clarity with a ratio of **24.22%** followed by **VS2** and **SI2**
* Only few have **I1** clarity with a ratio of **1.37%**.

In [ ]:
print("Let's have a look at the pricing distribution of diamond clarities:")
plt.subplots(figsize=(25, 10))

p=sns.violinplot(x=diamonds["clarity"],y=diamonds["price"],order=diamonds["clarity"].value_counts().index,palette=["#3f4f45","#557260","#5e9880","#a2c0a6","#f5a126","#e57716","#ea5b17","#6c3938"],saturation=1,linewidth=4,edgecolor="black")
p.axes.set_title("\nDiamond Pricing On Clarity\n",fontsize=30)
p.axes.set_xlabel("\nClarity",fontsize=25)
p.axes.set_ylabel("Price",fontsize=25)
p.set_yticklabels(p.get_yticks(), size = 20)
p.set_xticklabels(diamonds["clarity"].value_counts().index, size = 25)

sns.despine(left=True, bottom=True)
plt.show()

<a id="10"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Diamond Weights (in Carat)</p>

In [ ]:
print(f"Let's have a look at the distribution of diamond weights:")
plt.subplots(figsize=(20, 8))
p = sns.histplot(diamonds["carat"],color="#3f4f45",kde=True,bins=30,alpha=1,fill=True,edgecolor="black",linewidth=3)
p.axes.lines[0].set_color("orange")
p.axes.set_title("\nDiamond Weight Distribution\n",fontsize=25)
plt.ylabel("Count",fontsize=20)
plt.xlabel("\nWeight (in Carat)",fontsize=20)
plt.xlim(0, 3)
# plt.yscale("linear")
sns.despine(left=True, bottom=True)

plt.show()

In [ ]:
print("Let's have a look at the price distribution of weights:")

_, axes = plt.subplots(figsize=(20,8))
sns.kdeplot(y=diamonds["carat"], x=diamonds["price"],edgecolor="#1c1c1c",fill=True, kind="kde",shade=True,height=10,color="#3f4f45")
axes.set_title("\nPrice Distribution Of Weights\n",fontsize=25)
axes.set_xlabel("\nPrice",fontsize=20)
axes.set_ylabel("Weights In Carat",fontsize=20)
axes.set_ylim(0, 3)
    
sns.despine(left=True, bottom=True)
plt.show()

<a id="11"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Diamond's Depth Percentage</p>

In [ ]:
print(f"Let's have a look at the distribution of depth percentages:")
plt.subplots(figsize=(20, 8))
p = sns.histplot(diamonds["depth"],color="#5e9880",kde=True,bins=30,alpha=1,fill=True,edgecolor="black",linewidth=3)
p.axes.lines[0].set_color("orange")
p.axes.set_title("\nDiamond Depth Percentage Distribution\n",fontsize=25)
plt.ylabel("Count",fontsize=20)
plt.xlabel("\nDepth Percentage",fontsize=20)
plt.xlim(50, 75)
# plt.yscale("linear")
sns.despine(left=True, bottom=True)

plt.show()

In [ ]:
print("Let's have a look at the price distribution of depth percentage :")

_, axes = plt.subplots(figsize=(20,8))
sns.kdeplot(y=diamonds["depth"], x=diamonds["price"],edgecolor="#1c1c1c",fill=True, kind="kde",shade=True,height=10,color="#5e9880")
axes.set_title("\nPrice Distribution Of Depth Percentage\n",fontsize=25)
axes.set_xlabel("\nPrice",fontsize=20)
axes.set_ylabel("Depth Percentage",fontsize=20)
axes.set_ylim(50, 75)

sns.despine(left=True, bottom=True)
plt.show()

<a id="12"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Diamond's Table</p>

In [ ]:
print(f"Let's have a look at the distribution of diamond tables:")
plt.subplots(figsize=(20, 8))
p = sns.histplot(diamonds["table"],color="#6c3938",kde=True,bins=30,alpha=1,fill=True,edgecolor="black",linewidth=3)
p.axes.lines[0].set_color("orange")
p.axes.set_title("\nDiamond Tables Distribution\n",fontsize=25)
plt.ylabel("Count",fontsize=20)
plt.xlabel("\nDiamond's Table",fontsize=20)
plt.xlim(50, 75)
# plt.yscale("linear")
sns.despine(left=True, bottom=True)

plt.show()

In [ ]:
print("Let's have a lookat the price distribution of diamond tables:")

_, axes = plt.subplots(figsize=(20,8))
sns.kdeplot(y=diamonds["table"], x=diamonds["price"],edgecolor="#1c1c1c",fill=True, kind="kde",shade=True,height=10,color="#6c3938")
axes.set_title("\nPrice Distribution Of Diamond Tables\n",fontsize=25)
axes.set_xlabel("\nPrice",fontsize=20)
axes.set_ylabel("Diamond's Table",fontsize=20)
axes.set_ylim(50, 75)
    
sns.despine(left=True, bottom=True)
plt.show()

<a id="13"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Outlier Detection</p>

In [ ]:
sns.pairplot(diamonds, diag_kind = 'kde')

In [ ]:
plt.show()

**Insights:**

- The pairplot immediately tells us that there are some features with diamondspoints that are far from the rest of its colleagues. This will affect the outcome of our regression model and hence they will be removed.
* Let's examine the regression line in these distributions as well.

In [ ]:
_, axs = plt.subplots(2,3,figsize=(25,12),sharex=True)
plt.tight_layout(pad=4.0)

sns.regplot(x="price", y="x", data=diamonds, ax=axs[0,0], color="#3f4f45", fit_reg=True, line_kws=dict(color= "orange"))
axs[0,0].set_title("Price vs Length\n",fontsize=25)
axs[0,0].set_ylabel("x",fontsize=20)

sns.regplot(x="price", y="y", data=diamonds, ax=axs[0,1], color="#5e9880", fit_reg=True, line_kws=dict(color= "orange"))
axs[0,1].set_title("Price vs Width\n",fontsize=25)
axs[0,1].set_ylabel("y",fontsize=20)

sns.regplot(x="price", y="z", data=diamonds, ax=axs[0,2], color="#a2c0a6", fit_reg=True, line_kws=dict(color= "orange"))
axs[0,2].set_title("Price vs Depth\n",fontsize=25)
axs[0,2].set_ylabel("z",fontsize=20)

sns.regplot(x="price", y="depth", data=diamonds, ax=axs[1,0], color="#f5a126", fit_reg=True, line_kws=dict(color= "#6c3938"))
axs[1,0].set_title("Price vs Depth Percentage\n",fontsize=25)
axs[1,0].set_xlabel("\nPrice",fontsize=20)
axs[1,0].set_ylabel("depth",fontsize=20)

sns.regplot(x="price", y="table", data=diamonds, ax=axs[1,1], color="#ea5b17", fit_reg=True, line_kws=dict(color= "#6c3938"))
axs[1,1].set_title("Price vs Table\n",fontsize=25)
axs[1,1].set_xlabel("\nPrice",fontsize=20)
axs[1,1].set_ylabel("Table",fontsize=20)

sns.regplot(x="price", y="carat", data=diamonds, ax=axs[1,2], color="#6c3938", fit_reg=True, line_kws=dict(color= "orange"))
axs[1,2].set_title("Price vs Weight\n",fontsize=25)
axs[1,2].set_xlabel("\nPrice",fontsize=20)
axs[1,2].set_ylabel("carat",fontsize=20)

plt.suptitle("Regression Line",fontsize=30, y=1.03)
sns.despine(left=True, bottom=True)
plt.show()

In [ ]:
# Manually removing the outliers

diamonds = diamonds[(diamonds["x"]<10)&(diamonds["x"]>3)]
diamonds = diamonds[(diamonds["y"]<13)&(diamonds["y"]>2)]
diamonds = diamonds[(diamonds["z"]<6)&(diamonds["z"]>2)]
diamonds = diamonds[(diamonds["depth"]<73)&(diamonds["depth"]>53)]
diamonds = diamonds[(diamonds["table"]<71)&(diamonds["table"]>50)]
diamonds = diamonds[(diamonds["carat"]<3)]

print(f"After dropping the outliers, let's have a look at the new pairwise relationships:")


In [ ]:
sns.set()

In [ ]:
sns.pairplot(diamonds, diag_kind = 'kde')
plt.show()

<a id="14"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Correlation Map</p>

In [ ]:
corr = diamonds.corr()
corr

In [ ]:
catcol = ["color","clarity","cut"]
le = LabelEncoder()
for col in catcol:
        diamonds[col] = le.fit_transform(diamonds[col])


plt.subplots(figsize =(10, 10))

sns.heatmap(diamonds.corr(), cmap = palette, square=True, cbar_kws=dict(shrink =.82), 
            annot=True, vmin=-1, vmax=1, linewidths=3,linecolor='#3f4f45',annot_kws=dict(fontsize =12))
plt.title("Pearson Correlation Of Features\n", fontsize=25)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.show()

<a id="14"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">A Further Look</p>

### Correlation 'price' and 'carat'

In [ ]:
plt.figure(figsize=(16, 8))
sns.scatterplot(data=diamonds, x="carat", y="price", edgecolor=None, alpha=0.6);
plt.show()

### Let's now look at how this diagram varies by associating other features

- Correlation 'price' and 'carat' associated to 'color'

In [ ]:
plt.figure(figsize=(16, 8))
sns.scatterplot(
    data=diamonds, x="carat", y="price", hue="color", edgecolor=None, alpha=0.8
)
plt.show()

- Correlation 'price' and 'carat' associated to 'clarity'

In [ ]:
plt.figure(figsize=(16, 8))
sns.scatterplot(
    data=diamonds, x="carat", y="price", hue="clarity", edgecolor=None, alpha=0.8
)
plt.show()

- Correlation 'price' and 'carat' associated to 'cut'

In [ ]:
plt.figure(figsize=(16, 8))
sns.scatterplot(
    data=diamonds, x="carat", y="price", hue="cut", edgecolor=None, alpha=0.8
)
plt.show()

### Logarithmic base transformation

In [ ]:
diamonds["carat_log"] = np.log(diamonds["carat"])
diamonds["price_log"] = np.log(diamonds["price"])
diamonds

- Correlation 'price_log' and 'carat_log' associated to 'color'


In [ ]:
plt.figure(figsize=(16, 8))
sns.scatterplot(
    data=diamonds, x="carat_log", y="price_log", hue="color", edgecolor=None, alpha=0.8
)
plt.show()

- Correlation 'price_log' and 'carat_log' associated to 'clarity'


In [ ]:
plt.figure(figsize=(16, 8))
sns.scatterplot(
    data=diamonds, x="carat_log", y="price_log", hue="clarity", edgecolor=None, alpha=0.8
)
plt.show()

- Correlation 'price_log' and 'carat_log' associated to 'cut'

In [ ]:
plt.figure(figsize=(16, 8))
sns.scatterplot(
    data=diamonds, x="carat_log", y="price_log", hue="cut", edgecolor=None, alpha=0.8
)
plt.show()

Creating dictionaries with numeric values for categories 'cut', 'color' and 'clarity'

In [ ]:
# 'cut' quality range varies from 1 ('Fair') to 5 ('Ideal')
cut_quality = {"Fair": 1, "Good": 2, "Very Good": 3, "Premium": 4, "Ideal": 5}

# 'color' quality range varies from 1 ('J') to 7 ('D')
color_quality = {"D": 7, "E": 6, "F": 5, "G": 4, "H": 3, "I": 2, "J": 1}

# 'clarity' quality range varies from 1 ('I1') to 7 ('IF')
clarity_quality = {
    "I1": 1,
    "SI2": 2,
    "SI1": 3,
    "VS2": 4,
    "VS1": 5,
    "VVS2": 5,
    "VVS1": 6,
    "IF": 7,
}

In [ ]:
# # Applying (here's a good opportunity to use lambda functions!)
# diamonds["cut_quality"] = diamonds["cut"].apply(lambda x: cut_quality[x])
# diamonds["color_quality"] = diamonds["color"].apply(lambda x: color_quality[x])
# diamonds["clarity_quality"] = diamonds["clarity"].apply(lambda x: clarity_quality[x])
# diamonds

<a id="15"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Model Creation & Performance Evaluation</p>

After the standardization, we will split the dataset with a ratio of 0.2 i.e. 80% of data will be used for training and 20% for the validation process.

In [ ]:
sc = StandardScaler()
x = diamonds.drop(["price","cut","color","clarity"],axis =1) # price will be our target (y)
x = sc.fit_transform(x)
y = diamonds["price"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

<center> <span style="font-family:newtimeroman"> <span style="padding:5px;color:white;display:fill;border-radius:20px 50px;background-color:#FDD667;font-size:200%;font-weight: 500;color:#111423;overflow:hidden;">ㅤLinear Regressionㅤ</span>

In [ ]:
lr = LinearRegression()
lr.fit(x_train,y_train)
y_pred = lr.predict(x_test)

print("After performing the Linear Regression with SKLearn,\n")
print(f"R Squared Value: {metrics.r2_score(y_test,y_pred)}")
print(f"Adjusted R Squared Value: {1-(1-metrics.r2_score(y_test,y_pred))*(len(y_test)-1)/(len(y_test)-x_test.shape[1]-1)}")
print(f"Mean Absolute Error: {metrics.mean_absolute_error(y_test,y_pred)}")
print(f"Mean Squared Error: {metrics.mean_squared_error(y_test,y_pred)}")
print(f"Root Mean Squared Error: {metrics.mean_squared_error(y_test,y_pred,squared = False)}")

In [ ]:
sc = StandardScaler()
x = diamonds.drop(["price","cut","color","clarity"],axis =1) # price will be our target (y)
x = sc.fit_transform(x)
y = diamonds["price"]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
sns.set()

In [ ]:
plt.figure(figsize=(8,6))
# plt.xlabel('price')
# plt.ylabel('carat')
plt.scatter(diamonds['carat'], y, color='red', label='observed')
sns.despine(left=True, bottom=True)
# plot the predicted values together with the observed values
plt.scatter(x_test[:,0], y_pred, label='predicted')

plt.legend()
plt.show()

<center> <span style="font-family:newtimeroman"> <span style="padding:5px;color:white;display:fill;border-radius:20px 50px;background-color:#FDD667;font-size:200%;font-weight: 500;color:#111423;overflow:hidden;">ㅤDecision Tree Regressionㅤ</span>

In [ ]:
dt = DecisionTreeRegressor()
dt.fit(x_train,y_train)
y_pred = dt.predict(x_test)

print("After Performing Decision Tree Regression,\n")
print(f"R Squared Value: {metrics.r2_score(y_test,y_pred)}")
print(f"Adjusted R Squared Value: {1-(1-metrics.r2_score(y_test,y_pred))*(len(y_test)-1)/(len(y_test)-x_test.shape[1]-1)}")
print(f"Mean Absolute Error: {metrics.mean_absolute_error(y_test,y_pred)}")
print(f"Mean Squared Error: {metrics.mean_squared_error(y_test,y_pred)}")
print(f"Root Mean Squared Error: {metrics.mean_squared_error(y_test,y_pred,squared = False)}")

<center> <span style="font-family:newtimeroman"> <span style="padding:5px;color:white;display:fill;border-radius:20px 50px;background-color:#FDD667;font-size:200%;font-weight: 500;color:#111423;overflow:hidden;">ㅤRandom Forest Regressionㅤ</span>

In [ ]:
rf = RandomForestRegressor()
rf.fit(x_train,y_train)
y_pred = rf.predict(x_test)

print("After Performing Random Forest Regression,\n")
print(f"R Squared Value: {metrics.r2_score(y_test,y_pred)}")
print(f"Adjusted R Squared Value: {1-(1-metrics.r2_score(y_test,y_pred))*(len(y_test)-1)/(len(y_test)-x_test.shape[1]-1)}")
print(f"Mean Absolute Error: {metrics.mean_absolute_error(y_test,y_pred)}")
print(f"Mean Squared Error: {metrics.mean_squared_error(y_test,y_pred)}")
print(f"Root Mean Squared Error: {metrics.mean_squared_error(y_test,y_pred,squared = False)}")

<center> <span style="font-family:newtimeroman"> <span style="padding:5px;color:white;display:fill;border-radius:20px 50px;background-color:#FDD667;font-size:200%;font-weight: 500;color:#111423;overflow:hidden;">ㅤK-Neighbors Regressionㅤ</span>

In [ ]:
kn = KNeighborsRegressor()
kn.fit(x_train,y_train)
y_pred = kn.predict(x_test)

print("After Performing K-Neighbors Regression,\n")
print(f"R Squared Value: {metrics.r2_score(y_test,y_pred)}")
print(f"Adjusted R Squared Value: {1-(1-metrics.r2_score(y_test,y_pred))*(len(y_test)-1)/(len(y_test)-x_test.shape[1]-1)}")
print(f"Mean Absolute Error: {metrics.mean_absolute_error(y_test,y_pred)}")
print(f"Mean Squared Error: {metrics.mean_squared_error(y_test,y_pred)}")
print(f"Root Mean Squared Error: {metrics.mean_squared_error(y_test,y_pred,squared = False)}")

In [ ]:
x_train.shape

In [ ]:
x_test.shape

This first analysis indicates that a Random Forest Regressor might yield better results with this type of dataset. Now we can test it with Rick's diamonds.

In [ ]:
x_train

In [ ]:
rick_diamonds = pd.read_csv('rick_diamonds.csv')

In [ ]:
# diamonds.to_csv('diamonds_clean')

# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Model Explainability with SHAP (SHapley Additive exPlanations)</p>

AI solutions are somewhat mystic in nature to non-technical personel due to their "black box" nature. SHAP comes in handy for model explainability. It can break down the mechanics of any machine learning model and deep neural net to make them understandable to anyone. It is a Python package based on the 2016 NIPS paper about SHAP values. The premise of this paper and Shapley values comes from approaches in game theory. In a nutshel, SHAP offers two advantages: global model interpretability and local interpretability.

In [ ]:
diamonds = pd.read_csv('diamonds_clean',index_col=False)

In [ ]:
diamonds.head()

In [ ]:
diamonds.columns

In [ ]:
X, y = diamonds.drop(["Unnamed: 0","price", "carat_log", "price_log"], axis=1), diamonds["price"].values

In [ ]:
X

In [ ]:
y

- We could use the Ordinal Encoder to deal automatically with the categorical attributes if we wanted to. But I had already dealt with this issue manually

In [ ]:
# # Encode cats
# oe = OrdinalEncoder()
# cats = X.select_dtypes(exclude=np.number).columns.tolist()

In [ ]:
# cats

In [ ]:
# X.loc[:, cats] = oe.fit_transform(X[cats])

In [ ]:
X

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape, X_valid.shape

Lets look again at the Random Forest regression model from the SHAP perspective

In [ ]:
rf.fit(X_train,y_train)
y_pred = dt.predict(X_valid)

In [ ]:
rmse = mean_squared_error(y_valid, y_pred, squared=False)
rmse

Now, let's finally take a peek behind the curtains and calculate the Shapley values for the training set.

We start by creating an explainer object for our model:

In [ ]:
dt_explainer = shap.TreeExplainer(dt, X_train, feature_names=X_train.columns.tolist())

TreeExplainer is a special class of SHAP, optimized to work with any tree-based model in Sklearn, XGBoost, LightGBM, CatBoost, and so on.
You can use KernelExplainer for any other type of model, though it is slower than tree explainers.

In [ ]:
# dt_explainer

This tree explainer has many methods, one of which is shap_values:

In [ ]:
# Shap values with tree explainer
dt_explainer.shap_values(X_train, y_train) # this breaks the kernel everytime..

In [ ]:
# !pip install xgboost

In [ ]:
import xgboost as xgb

As a baseline, we fit an XGBRegressor model and evaluate the performance with Root Mean Squared Error:

In [ ]:
model = xgb.XGBRegressor(n_estimators=1000, tree_method="gpu_hist").fit(X_train, y_train)

In [ ]:
preds = model.predict(X_valid)
rmse = mean_squared_error(y_valid, preds, squared=False)
rmse

In [ ]:
# Shap values with XGBoost core model
booster_xgb = model.get_booster()
shap_values_xgb = booster_xgb.predict(xgb.DMatrix(X_train, y_train), pred_contribs=True)

In [ ]:
shap_values_xgb.shape

But wait - the Shap values from the tree explainer had nine columns; this one has 10! Don't worry; we can safely ignore the last column for now, as it just contains the bias term which XGBoost adds by default:

In [ ]:
shap_values_xgb = shap_values_xgb[:, :-1]

pd.DataFrame(shap_values_xgb, columns=X_train.columns.tolist()).head()

We got the Shapley values; now what? Now, we get plottin'.

In [ ]:
shap.summary_plot(shap_values_xgb, X_train, feature_names=X_train.columns, plot_type="bar")

The carat stands out as the driving factor for a diamond's price. Reading the axis title below, we see that the importances are just the average absolute Shapley values for a feature. We can check that below:

In [ ]:
pd.DataFrame(shap_values_xgb, columns=X_train.columns)["carat"].abs().mean()

We won't also stop here. In the above plots, we only looked at absolute values of importance. We don't know which feature positively or negatively influences the model. Let's do that with SHAP summary_plot:

In [ ]:
shap.summary_plot(shap_values_xgb, X_train, feature_names=X_train.columns);

- The left vertical axis denotes feature names, ordered based on importance from top to bottom.
- The horizontal axis represents the magnitude of the SHAP values for predictions.
- The vertical right axis represents the actual magnitude of a feature as it appears in the dataset and colors the points.
- We see that as carat increases, its effect on the model is more positive. The same is true for y feature. The x and z features are a bit tricky with a cluster of mixed points around the center.

# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Exploring each feature with dependence plots</p>

We can get a deeper insight into each feature's effect on the entire dataset with dependence plots. Let's see an example and explain it later:

In [ ]:
shap.dependence_plot("carat", shap_values_xgb, X_train, interaction_index=None)

This plot aligns with what we saw in the summary plot before. As carat increases, its SHAP value increases. By changing the interaction_index parameter to auto, we can color the points with a feature that most strongly interacts with carat:

In [ ]:
shap.dependence_plot("carat", shap_values_xgb, X_train, interaction_index="auto");

It seems that the carat interacts with the clarity of the diamonds much stronger than other features.

Let's now create a dependence plot for categorical features:

In [ ]:
shap.dependence_plot("color", shap_values_xgb, X_train, feature_names=X_train.columns);

This plot also goes in hand with the summary plot. The latest color categories affect the prices negatively while interacting with carat.

In [ ]:
shap.dependence_plot("clarity", shap_values_xgb, X_train, feature_names=X_train.columns);

In [ ]:
shap.dependence_plot("cut", shap_values_xgb, X_train, feature_names=X_train.columns);

In [ ]:
shap.dependence_plot("depth", shap_values_xgb, X_train, feature_names=X_train.columns);

Feature Interactions with Shapley values

One of the most fantastic attributes of SHAP and Shapley values is their ability to find relationships between features accurately. We have already got a taste of that in the last section when SHAP found the most robust interacting feature in dependence plots.

We can go a step further and find all feature interactions ordered by their interaction strength. For that, we need a different set of values - SHAP interaction values.

They can be calculated using the shap_interaction_values of the tree explainer object like so:

In [ ]:
interactions_xgb = booster_xgb.predict(
    xgb.DMatrix(X_train, y_train), pred_interactions=True
)

By setting pred_interactions to True, we get SHAP interaction values in only 12 seconds. It is a 3D array, with the last column axes being the bias terms:

In [ ]:
interactions_xgb.shape

In [ ]:
def get_top_k_interactions(feature_names, shap_interactions, k):
    # Get the mean absolute contribution for each feature interaction
    aggregate_interactions = np.mean(np.abs(shap_interactions[:, :-1, :-1]), axis=0)
    interactions = []
    for i in range(aggregate_interactions.shape[0]):
        for j in range(aggregate_interactions.shape[1]):
            if j < i:
                interactions.append(
                    (
                        feature_names[i] + "-" + feature_names[j],
                        aggregate_interactions[i][j] * 2,
                    )
                )
    # sort by magnitude
    interactions.sort(key=lambda x: x[1], reverse=True)
    interaction_features, interaction_values = map(tuple, zip(*interactions))

    return interaction_features[:k], interaction_values[:k]


top_10_inter_feats, top_10_inter_vals = get_top_k_interactions(
    X_train.columns, interactions_xgb, 10
)

Now, top_10_inter_feats contains 10 of the strongest interactions between all possible pairs of features:

In [ ]:
top_10_inter_feats

We can create another function that plots these pairs based on their interaction strengths:

In [ ]:
def plot_interaction_pairs(pairs, values):
    plt.bar(pairs, values)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show();

In [ ]:
top_10_inter_feats, top_10_inter_vals = get_top_k_interactions(
    X_train.columns, interactions_xgb, 10
)

plot_interaction_pairs(top_10_inter_feats, top_10_inter_vals)

# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Predicting the price of Ricks' diamonds</p>

We know that the price is right-skewed, and thus we checked to see if a log transformation could make the Price attribute approximately normal to give fighting chances to algorithms that assume normality. We never actually plotted it though, let's do it now:

In [ ]:
fig = px.histogram(diamonds, x=["price_log"], title = 'Histgram of Log Price', template = 'plotly_dark')
fig.show()

It's not exactly normal... Still, it's what we got! Let's see the outcome.
- Also, we should check what Shapiro-Wilks has to say about this!

In [ ]:
diamonds.to_csv('diamonds_clean.csv')

In [ ]:
diamonds.head()

In [ ]:
def cria_modelos(df, df_pred, x_test, y_test):
    '''
    Função que cria modelos para a previsão
    '''
    X = df[x_test]
    y = df[y_test]

    model = GradientBoostingRegressor()
    model.fit(X,y)
    
    price_pred = model.predict(df_pred[x_test])
    
    return price_pred

In [ ]:
diamonds_test = diamonds

In [ ]:
diamonds_rick = pd.read_csv("rick_diamonds.csv")

Data transformation

In [ ]:
diamonds_test.loc[diamonds_test['cut'] == 'Fair', 'cut'] = 1
diamonds_test.loc[diamonds_test['cut'] == 'Good', 'cut'] = 2
diamonds_test.loc[diamonds_test['cut'] == 'Very Good', 'cut'] = 3
diamonds_test.loc[diamonds_test['cut'] == 'Premium', 'cut'] = 4
diamonds_test.loc[diamonds_test['cut'] == 'Ideal', 'cut'] = 5

diamonds_rick.loc[diamonds_rick['cut'] == 'Fair', 'cut'] = 1
diamonds_rick.loc[diamonds_rick['cut'] == 'Good', 'cut'] = 2
diamonds_rick.loc[diamonds_rick['cut'] == 'Very Good', 'cut'] = 3
diamonds_rick.loc[diamonds_rick['cut'] == 'Premium', 'cut'] = 4
diamonds_rick.loc[diamonds_rick['cut'] == 'Ideal', 'cut'] = 5

#transformação dos dados para a melhor previsão do modelo
diamonds_test[['carat', 'cut','depth','x','y','z','table','price']].applymap(lambda x: np.log10(x + 1))
diamonds_rick[['carat', 'cut','depth','x','y','z','table']].applymap(lambda x: np.log10(x + 1))

Modeling

In [ ]:
#Criando a Coluna price_predicted no dataframe
diamonds_rick['price_predicted'] = 0 

In [ ]:
#Masks para as modelagens

#Masks de Clarity
# df de testes
teste_if = diamonds_test['clarity'] == 'IF'
teste_vvs1 = diamonds_test['clarity'] == 'VVS1'
teste_vvs2 = diamonds_test['clarity'] == 'VVS2'
teste_vs1 = diamonds_test['clarity'] == 'VS1'
teste_vs2 = diamonds_test['clarity'] == 'VS2'
teste_si1 = diamonds_test['clarity'] == 'SI1'
teste_si2 = diamonds_test['clarity'] == 'SI2'
teste_i1 = diamonds_test['clarity'] == 'I1'

# df do rick
rick_if = diamonds_rick['clarity'] == 'IF'
rick_vvs1 = diamonds_rick['clarity'] == 'VVS1'
rick_vvs2 = diamonds_rick['clarity'] == 'VVS2'
rick_vs1 = diamonds_rick['clarity'] == 'VS1'
rick_vs2 = diamonds_rick['clarity'] == 'VS2'
rick_si1 = diamonds_rick['clarity'] == 'SI1'
rick_si2 = diamonds_rick['clarity'] == 'SI2'
rick_i1 = diamonds_rick['clarity'] == 'I1'

#Masks de Color
#df de testes
teste_d = diamonds_test['color'] == 'D'
teste_e = diamonds_test['color'] == 'E'
teste_f = diamonds_test['color'] == 'F'
teste_g = diamonds_test['color'] == 'G'
teste_h = diamonds_test['color'] == 'H'
teste_i = diamonds_test['color'] == 'I'
teste_j = diamonds_test['color'] == 'J'
#df do rick
rick_d = diamonds_rick['color'] == 'D'
rick_e = diamonds_rick['color'] == 'E'
rick_f = diamonds_rick['color'] == 'F'
rick_g = diamonds_rick['color'] == 'G'
rick_h = diamonds_rick['color'] == 'H'
rick_i = diamonds_rick['color'] == 'I'
rick_j = diamonds_rick['color'] == 'J'

In [ ]:
#listas clarity mask
teste_clarity_list = [
    teste_if, teste_vvs1, teste_vvs2, teste_vs1, teste_vs2, teste_si1,
    teste_si2, teste_i1
]

rick_clarity_list = [
    rick_if, rick_vvs1, rick_vvs2, rick_vs1, rick_vs2, rick_si1, rick_si2,
    rick_i1
]
#listas color mask
teste_color_list = [
    teste_d, teste_e, teste_f, teste_g, teste_h, teste_i, teste_j
]

rick_color_list = [rick_d, rick_e, rick_f, rick_g, rick_h, rick_i, rick_j]

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
#Loop para criar todas as combinações de modelos 
for clarity in list(zip(teste_clarity_list, rick_clarity_list)):
    for color in list(zip(teste_color_list, rick_color_list)):
        diamonds_rick.loc[clarity[1] & color[1],
                          'price_predicted'] = cria_modelos(
                              diamonds_test[clarity[0] & color[0]],
                              diamonds_rick[clarity[1] & color[1]],
                              ['carat', 'cut','depth','x','y','z','table'], 'price')

In [ ]:
diamonds_test[['carat', 'cut','depth','x','y','z','table','price']].applymap(lambda x: round(np.exp(x),ndigits=2))
diamonds_rick[['carat', 'cut','depth','x','y','z','table', 'price_predicted']].applymap(lambda x: round(np.exp(x),ndigits=2))

diamonds_rick.to_csv('price_pred.csv', index=False)
diamonds_rick

- This process yields an RMSE of just $579.42, way below our second objective,
	"- to progressively train and test a regression model until its accuracy meet a certain standard (defined by the RMSE). Rick’s goal is to obtain an average error below 900 dollars."

- Rick is now a richer man and we have proved the usefulness of machine learning! 🥳 🥂 🍾

<a id="16"></a>
# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Hope this was useful!</p>

<p>


<b>👉Shoot me mails :</b> hbatistuzzo@gmail.com<br>
<b>👉Connect on LinkedIn :</b> https://www.linkedin.com/in/hbatistuzzo <br>
<b>👉Explore Github :</b> https://github.com/hbatistuzzo    
    
</p> 

# <p style="padding:10px;background-color:#0fa79f;margin:0;color:#111423;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500">Backup section of old ideas and tests</p>

BACKUP - DATA TRANSFORMATION

In [ ]:
diamonds_new = diamonds
diamonds_new.loc[diamonds['cut'] == 'Fair','cut'] = 1
diamonds_new.loc[diamonds['cut'] == 'Good','cut'] = 2
diamonds_new.loc[diamonds['cut'] == 'Very Good','cut'] = 3
diamonds_new.loc[diamonds['cut'] == 'Premium','cut'] = 4
diamonds_new.loc[diamonds['cut'] == 'Ideal','cut'] = 5

In [ ]:
diamonds_new

NEW MODEL BASED ON CUTS

In [ ]:
X = diamonds_new[['carat','cut']]
y = diamonds_new['price']

In [ ]:
diamonds_new[['carat','cut']]

In [ ]:
model = LinearRegression()
model.fit(X,y)

In [ ]:
rick_diamonds = pd.read_csv('rick_diamonds.csv')
rick_diamonds

In [ ]:
rick_diamonds.loc[rick_diamonds['cut'] == 'Good','cut'] = 2
rick_diamonds.loc[rick_diamonds['cut'] == 'Very Good','cut'] = 3
rick_diamonds.loc[rick_diamonds['cut'] == 'Fair','cut'] = 1
rick_diamonds.loc[rick_diamonds['cut'] == 'Premium','cut'] = 4
rick_diamonds.loc[rick_diamonds['cut'] == 'Ideal','cut'] = 5

In [ ]:
y_predicted_rick = model.predict(rick_diamonds[['carat','cut']])

In [ ]:
rick_diamonds['price_predicted'] = y_predicted_rick

In [ ]:
rick_diamonds

In [ ]:
rick_diamonds.to_csv('rick_diamonds_pred.csv')

Now with clarity

In [ ]:
rick_diamonds

In [ ]:
rick_diamonds.loc[rick_diamonds['cut'] == 'Good','cut'] = 2
rick_diamonds.loc[rick_diamonds['cut'] == 'Very Good','cut'] = 3
rick_diamonds.loc[rick_diamonds['cut'] == 'Fair','cut'] = 1
rick_diamonds.loc[rick_diamonds['cut'] == 'Premium','cut'] = 4
rick_diamonds.loc[rick_diamonds['cut'] == 'Ideal','cut'] = 5

In [ ]:
rick_diamonds

In [ ]:
diamonds_new.loc[diamonds['cut'] == 'Fair','cut'] = 1
diamonds_new.loc[diamonds['cut'] == 'Good','cut'] = 2
diamonds_new.loc[diamonds['cut'] == 'Very Good','cut'] = 3
diamonds_new.loc[diamonds['cut'] == 'Premium','cut'] = 4
diamonds_new.loc[diamonds['cut'] == 'Ideal','cut'] = 5

In [ ]:
diamonds_new

In [ ]:
diamonds_new.loc[diamonds['color'] == 'J','color'] = 1
diamonds_new.loc[diamonds['color'] == 'I','color'] = 2
diamonds_new.loc[diamonds['color'] == 'H','color'] = 3
diamonds_new.loc[diamonds['color'] == 'G','color'] = 4
diamonds_new.loc[diamonds['color'] == 'F','color'] = 5
diamonds_new.loc[diamonds['color'] == 'E','color'] = 6
diamonds_new.loc[diamonds['color'] == 'D','color'] = 7

In [ ]:
diamonds_new

In [ ]:
rick_diamonds.loc[rick_diamonds['color'] == 'J','color'] = 1
rick_diamonds.loc[rick_diamonds['color'] == 'I','color'] = 2
rick_diamonds.loc[rick_diamonds['color'] == 'H','color'] = 3
rick_diamonds.loc[rick_diamonds['color'] == 'G','color'] = 4
rick_diamonds.loc[rick_diamonds['color'] == 'F','color'] = 5
rick_diamonds.loc[rick_diamonds['color'] == 'E','color'] = 6
rick_diamonds.loc[rick_diamonds['color'] == 'D','color'] = 7

In [ ]:
rick_diamonds

In [ ]:
diamonds_new.loc[diamonds['clarity'] == 'I1','clarity'] = 1
diamonds_new.loc[diamonds['clarity'] == 'SI2','clarity'] = 2
diamonds_new.loc[diamonds['clarity'] == 'SI1','clarity'] = 3
diamonds_new.loc[diamonds['clarity'] == 'VS2','clarity'] = 4
diamonds_new.loc[diamonds['clarity'] == 'VS1','clarity'] = 5
diamonds_new.loc[diamonds['clarity'] == 'VVS2','clarity'] = 6
diamonds_new.loc[diamonds['clarity'] == 'VVS1','clarity'] = 7
diamonds_new.loc[diamonds['clarity'] == 'IF','clarity'] = 8

In [ ]:
rick_diamonds.loc[rick_diamonds['clarity'] == 'I1','clarity'] = 1
rick_diamonds.loc[rick_diamonds['clarity'] == 'SI2','clarity'] = 2
rick_diamonds.loc[rick_diamonds['clarity'] == 'SI1','clarity'] = 3
rick_diamonds.loc[rick_diamonds['clarity'] == 'VS2','clarity'] = 4
rick_diamonds.loc[rick_diamonds['clarity'] == 'VS1','clarity'] = 5
rick_diamonds.loc[rick_diamonds['clarity'] == 'VVS2','clarity'] = 6
rick_diamonds.loc[rick_diamonds['clarity'] == 'VVS1','clarity'] = 7
rick_diamonds.loc[rick_diamonds['clarity'] == 'IF','clarity'] = 8

In [ ]:
rick_diamonds

In [ ]:
rick_diamonds.to_csv('rick_diamonds_updated')

In [ ]:
diamonds_new.to_csv('diamonds_new_updated')

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

In [ ]:
X = diamonds_new[['carat','cut','color','clarity']]
y = diamonds_new['price']

model.fit(X,y)

y_predicted_rick = model.predict(rick_diamonds[['carat','cut','color','clarity']])


In [ ]:
rick_diamonds

In [ ]:
rick_diamonds['price_predicted'] = y_predicted_rick

In [ ]:
rick_diamonds.to_csv('rick_diamonds_pred.csv')

Error with mean: 3980.71
Error with 1st model (carat): 1605.15
Error with 2nd model (cut and carat): 1578.53

In [ ]:
def cria_modelos(df,df_pred,x_test,y_test):
    '''
    autoexplicativo convenhamos
    '''
    X = df[x_test] #variaveis que eu vou usar pra treinar
    y = df[y_test] #y que eu to usando pra treinar

    model = LinearRegression()
    model.fit(X,y)
    price_pred = model.predict(df_pred[x_test])

    return price_pred

In [ ]:
#criando a coluna price_predicted no dataframe
rick_diamonds['price_predicted'] = 0

In [ ]:
mask_test = (diamonds_test['clarity'] == 'VVS1') | (diamonds_test['clarity'] == 'VVS2') | (diamonds_test['clarity'] == 'IF')
mask_rick = (rick_diamonds['clarity'] == 'VVS1') | (rick_diamonds['clarity'] == 'VVS2') | (rick_diamonds['clarity'] == 'IF')

diamonds_rick.loc[mask_rick,'price_predicted'] = cria_modelos(diamonds_test[mask_teste],diamonds_rick[mask_rick],['carat'],['price'])